# Protego — Inference & Evaluation Demo

This notebook shows how to:
1. Load a pretrained **Privacy Protection Texture (PPT)** for a protectee.
2. Apply it to a sample image (pose-invariant 3D warping).
3. Evaluate the **before/after retrieval recall** on the preprocessed FaceScrub subset.

Make sure you have run `bash setup_quick.sh`, activated the `protego` environment, and downloaded the assets with `python -m tools.download_assets` (see the README).

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protego import BASE_PATH
from protego.utils import load_mask, load_imgs
from protego.UVMapping import UVGenerator

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Load the pretrained PPT and the UV mapper

The PPT is a universal mask in UV space saved as `univ_mask.npy` under `experiments/<exp_name>/<protectee>/`. The `UVGenerator` (SMIRK + FLAME + MediaPipe) deforms it to match the pose/expression of any face image of the user.

In [ ]:
protectee = 'Bradley_Cooper'
exp_name = 'default'
epsilon = 16 / 255.
use_bin_mask = True
three_d = True

mask_path = os.path.join(BASE_PATH, 'experiments', exp_name, protectee, 'univ_mask.npy')
assert os.path.exists(mask_path), f'PPT not found at {mask_path}. Download the demo PPT (DRIVE_LINK__DEMO_PPT).'
mask = load_mask(mask_path=mask_path, device=device)
print('Loaded PPT with shape', tuple(mask.shape))

smirk_base_path = os.path.join(BASE_PATH, 'smirk')
uvmapper = UVGenerator(
    smirk_ckpts_path=os.path.join(smirk_base_path, 'pretrained_models/SMIRK_em1.pt'),
    smirk_base_path=smirk_base_path,
    mp_ldmk_model_path=os.path.join(smirk_base_path, 'assets/face_landmarker.task'),
    device=device,
)

## 2. Apply the PPT to a sample image

The face image is assumed to be already cropped/aligned (as in the preprocessed FaceScrub subset). For uncropped images, use `tools/protect_imgs.py` with `need_detection=True`.

In [ ]:
@torch.no_grad()
def protect(face: torch.Tensor) -> torch.Tensor:
    """face: [1, 3, 224, 224], range [0, 1]. Returns the protected image in [0, 1]."""
    uv, bin_mask, _ = uvmapper.forward(imgs=face, align_ldmks=False, batch_size=-1)
    pert = F.grid_sample(mask, uv, align_corners=True, mode='bilinear') if three_d else mask
    if use_bin_mask:
        pert = pert * bin_mask
    pert = pert.clamp(-epsilon, epsilon)
    return (face + pert).clamp(0., 1.)

sample_dir = os.path.join(BASE_PATH, 'face_db', 'face_scrub', protectee)
sample_paths = sorted([os.path.join(sample_dir, f) for f in os.listdir(sample_dir)
                       if f.lower().endswith(('.png', '.jpg', '.jpeg')) and not f.startswith(('.', '_'))])
orig = load_imgs(img_paths=sample_paths[:1], img_sz=224, drange=255, device=device).div(255.)
prot = protect(orig)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig[0].permute(1, 2, 0).cpu().numpy()); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(prot[0].permute(1, 2, 0).cpu().numpy()); axes[1].set_title('Protected'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 2b. Apply the PPT to a sample video

Protego shines on video, where per-frame consistency keeps the result natural. We detect and crop the face in each frame, deform the PPT to that frame's pose/expression, blend the perturbation back into the original frame, and write a protected video. This reuses the same logic as `tools/protect_vids.py`.

In [ ]:
import cv2
from IPython.display import Video, display
from protego.FaceDetection import FD
from protego.utils import crop_face

# A short sample clip of the protectee. Swap in your own video if you like.
src_vid_path = os.path.join(BASE_PATH, 'face_db', 'vids', protectee, 'bc1_480p.mp4')
dst_vid_path = os.path.join(BASE_PATH, 'results', 'vids', protectee, 'protected_demo.mp4')
os.makedirs(os.path.dirname(dst_vid_path), exist_ok=True)
assert os.path.exists(src_vid_path), f'Sample video not found at {src_vid_path}. Provide one (DRIVE_LINK__SAMPLE_MEDIA).'

fd = FD(model_name='mtcnn', device=device)

@torch.no_grad()
def protect_video(src_path: str, dst_path: str, max_frames: int = 150) -> None:
    """Protect up to `max_frames` frames of a video and write the result to `dst_path`."""
    reader = cv2.VideoCapture(src_path)
    fps = reader.get(cv2.CAP_PROP_FPS)
    width = int(reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(reader.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(dst_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    frame_cnt = 0
    while frame_cnt < max_frames:
        ret, frame = reader.read()
        if not ret:
            break
        frame_cnt += 1
        # BGR uint8 -> RGB float [0,1], shape [3, H, W]
        frame_pt = torch.tensor(frame, dtype=torch.float32, device=device).permute(2, 0, 1).div(255.)[[2, 1, 0], :, :]
        cropped_face, face_pos = crop_face(img=frame_pt, detector=fd, crop_loosen=1., verbose=False, strict=False)
        if cropped_face is not None and face_pos is not None:
            face_pos = list(face_pos)
            fh, fw = cropped_face.shape[1], cropped_face.shape[2]
            resized = F.interpolate(cropped_face.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
            uv, bin_mask, _ = uvmapper.forward(imgs=resized, align_ldmks=False, batch_size=-1)
            pert = F.grid_sample(mask, uv, align_corners=True, mode='bilinear') if three_d else mask
            if use_bin_mask:
                pert = pert * bin_mask
            pert = F.interpolate(pert, size=(fh, fw), mode='bilinear', align_corners=False).clamp_(-epsilon, epsilon)
            protected_face = (cropped_face + pert.squeeze(0)).clamp_(0., 1.)
            frame_pt[:, face_pos[1]:face_pos[3], face_pos[0]:face_pos[2]] = protected_face.contiguous()
        # RGB float [0,1] -> BGR uint8 for writing
        out_frame = frame_pt.permute(1, 2, 0).mul(255.).to(torch.uint8)[:, :, [2, 1, 0]].cpu().contiguous().numpy()
        writer.write(out_frame)
        if frame_cnt % 50 == 0:
            print(f'Processed {frame_cnt} frames...')
    reader.release()
    writer.release()
    print(f'Done. Protected video saved to {dst_path}')

protect_video(src_vid_path, dst_vid_path)
display(Video(dst_vid_path, embed=True, width=360))

## 3. Evaluate retrieval recall (before vs. after protection)

We reuse the standard pipeline in eval mode. It deforms each protectee's PPT onto their evaluation images, builds the FaceScrub noise/retrieval gallery (augmented with the other protectees' protected features), and reports the retrieval recalls. Keys: `1b` is the unprotected baseline recall (should be high), `1a`/`2a`/`2b` are the protected recalls (lower is better).

In [ ]:
import math
from omegaconf import OmegaConf
from protego.utils import get_usable_img_paths
from protego.run_exp import run

configs = {
    'device': str(device),
    'exp_name': 'demo_eval',
    'need_cropping': False, 'fd_name': 'mtcnn', 'crop_loosen': 1., 'shuffle': False,
    'uv_gen_align_ldmk': False, 'uv_gen_batch': 8,
    'three_d': three_d, 'mask_size': 224, 'bin_mask': use_bin_mask, 'epsilon': epsilon,
    'mask_name': [exp_name, 'univ_mask.npy'], 'eval_db': 'face_scrub',
    'eval_fr_names': ['ir50_adaface_casia'],
    'train_fr_names': [], 'save_univ_mask': False, 'visualize_interval': 0,
    'query_portion': 0.5, 'vis_eval': True, 'lpips_backbone': 'vgg',
}
cfgs = OmegaConf.create(configs)

# Build the train/eval split for a couple of protectees that have a trained PPT.
train_data_path = os.path.join(BASE_PATH, 'face_db', cfgs.eval_db)
exp_dir = os.path.join(BASE_PATH, 'experiments', exp_name)
protectees = sorted([n for n in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, n))])[:2]
data = {}
for p in protectees:
    imgs = get_usable_img_paths(os.path.join(train_data_path, p))
    split = math.floor(len(imgs) * 0.6)
    data[p] = {'train': imgs[:split], 'eval': imgs[split:]}

run(cfgs, mode='eval', data=data)
print('Eval results saved under results/eval/demo_eval/<protectee>/')